# Phase 3b: Baseline Models
## DNA Gene Mapping Project
**Author:** Sharique Mohammad  
**Date:** February 2026  

---

## Objective
Train baseline ML models to establish performance benchmarks:
- **Logistic Regression** (linear baseline)
- **Decision Tree** (non-linear baseline)

## Metrics
- **Variant Pathogenicity**: F1-Score, Precision, Recall, ROC-AUC
- **SV Risk**: Recall (prioritize catching high-risk SVs)

## Deliverables
- Baseline model performance
- Feature importance rankings
- Performance floor for advanced models

---
## 1. Setup

In [ ]:
# Imports
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import json
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    roc_auc_score, roc_curve, precision_recall_curve,
    f1_score, precision_score, recall_score, accuracy_score
)

import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Imports successful")

In [ ]:
# Configuration
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data" / "ml"
MODEL_DIR = PROJECT_ROOT / "models"
METRICS_DIR = PROJECT_ROOT / "data" / "ml" / "metrics"
FIGURES_DIR = PROJECT_ROOT / "data" / "analytical" / "figures" / "phase3"

# Create directories
for dir_path in [METRICS_DIR, FIGURES_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

print("="*80)
print("PHASE 3B: BASELINE MODELS")
print("="*80)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Random state: {RANDOM_STATE}")
print("="*80)

---
## 2. Load Prepared Datasets

In [ ]:
print("Loading prepared datasets...")

# Load variant pathogenicity datasets
with open(DATA_DIR / "variant_train_original.pkl", 'rb') as f:
    train_orig = pickle.load(f)
    X_train_orig, y_train_orig = train_orig['X'], train_orig['y']

with open(DATA_DIR / "variant_train_balanced.pkl", 'rb') as f:
    train_bal = pickle.load(f)
    X_train_bal, y_train_bal = train_bal['X'], train_bal['y']

with open(DATA_DIR / "variant_validation.pkl", 'rb') as f:
    val_data = pickle.load(f)
    X_val, y_val = val_data['X'], val_data['y']

with open(DATA_DIR / "variant_test.pkl", 'rb') as f:
    test_data = pickle.load(f)
    X_test, y_test = test_data['X'], test_data['y']

print("\nVariant Pathogenicity Datasets:")
print(f"  Train (original): {X_train_orig.shape}")
print(f"  Train (balanced): {X_train_bal.shape}")
print(f"  Validation: {X_val.shape}")
print(f"  Test: {X_test.shape}")

# Load SV datasets
with open(DATA_DIR / "sv_train.pkl", 'rb') as f:
    sv_train = pickle.load(f)
    X_sv_train, y_sv_train = sv_train['X'], sv_train['y']

with open(DATA_DIR / "sv_validation.pkl", 'rb') as f:
    sv_val = pickle.load(f)
    X_sv_val, y_sv_val = sv_val['X'], sv_val['y']

with open(DATA_DIR / "sv_test.pkl", 'rb') as f:
    sv_test = pickle.load(f)
    X_sv_test, y_sv_test = sv_test['X'], sv_test['y']

print("\nStructural Variant Datasets:")
print(f"  Train: {X_sv_train.shape}")
print(f"  Validation: {X_sv_val.shape}")
print(f"  Test: {X_sv_test.shape}")

---
## 3. Helper Functions

In [ ]:
def evaluate_model(model, X_train, y_train, X_val, y_val, model_name, task_name):
    """
    Comprehensive model evaluation.
    
    Returns:
        Dictionary with all metrics
    """
    # Predictions
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    y_val_proba = model.predict_proba(X_val)[:, 1]
    
    # Metrics
    results = {
        'model': model_name,
        'task': task_name,
        'train': {
            'accuracy': accuracy_score(y_train, y_train_pred),
            'precision': precision_score(y_train, y_train_pred),
            'recall': recall_score(y_train, y_train_pred),
            'f1': f1_score(y_train, y_train_pred)
        },
        'validation': {
            'accuracy': accuracy_score(y_val, y_val_pred),
            'precision': precision_score(y_val, y_val_pred),
            'recall': recall_score(y_val, y_val_pred),
            'f1': f1_score(y_val, y_val_pred),
            'roc_auc': roc_auc_score(y_val, y_val_proba)
        }
    }
    
    # Print summary
    print(f"\n{model_name} - {task_name}")
    print("-" * 60)
    print(f"Train F1: {results['train']['f1']:.4f}")
    print(f"Val   F1: {results['validation']['f1']:.4f}")
    print(f"Val  AUC: {results['validation']['roc_auc']:.4f}")
    print(f"Val Prec: {results['validation']['precision']:.4f}")
    print(f"Val Rec:  {results['validation']['recall']:.4f}")
    
    return results, y_val_pred, y_val_proba

def plot_confusion_matrix(y_true, y_pred, model_name, task_name, save_path):
    """Plot and save confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'{model_name} - {task_name}\nConfusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    
def plot_roc_curve(y_true, y_proba, model_name, task_name, save_path):
    """Plot and save ROC curve"""
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    auc = roc_auc_score(y_true, y_proba)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'{model_name} (AUC = {auc:.3f})', linewidth=2)
    plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} - {task_name}\nROC Curve')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

def plot_feature_importance(model, feature_names, model_name, task_name, save_path, top_n=20):
    """Plot and save feature importance"""
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    elif hasattr(model, 'coef_'):
        importances = np.abs(model.coef_[0])
    else:
        print(f"  No feature importance for {model_name}")
        return
    
    # Get top features
    indices = np.argsort(importances)[-top_n:]
    top_features = [feature_names[i] for i in indices]
    top_importances = importances[indices]
    
    # Plot
    plt.figure(figsize=(10, 8))
    plt.barh(range(len(top_features)), top_importances)
    plt.yticks(range(len(top_features)), top_features)
    plt.xlabel('Importance')
    plt.title(f'{model_name} - {task_name}\nTop {top_n} Features')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    return list(zip(top_features, top_importances))

print("Helper functions defined")

---
## 4. Task 1: Variant Pathogenicity - Logistic Regression

In [ ]:
print("="*80)
print("TASK 1: VARIANT PATHOGENICITY PREDICTION")
print("="*80)
print("\nModel 1: Logistic Regression (L2 regularization)")
print("-" * 60)

# Train on balanced data
lr_model = LogisticRegression(
    penalty='l2',
    C=1.0,
    max_iter=1000,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("Training...")
lr_model.fit(X_train_bal, y_train_bal)
print("Training complete")

# Evaluate
lr_results, lr_pred, lr_proba = evaluate_model(
    lr_model, X_train_bal, y_train_bal, X_val, y_val,
    "Logistic Regression", "Variant Pathogenicity"
)

# Save model
lr_model_file = MODEL_DIR / "baseline_lr_variants.pkl"
with open(lr_model_file, 'wb') as f:
    pickle.dump(lr_model, f)
print(f"\n Model saved: {lr_model_file.name}")

In [ ]:
# Visualizations
print("\nGenerating visualizations...")

# Confusion matrix
plot_confusion_matrix(
    y_val, lr_pred, "Logistic Regression", "Variant Pathogenicity",
    FIGURES_DIR / "01_lr_variants_confusion_matrix.png"
)

# ROC curve
plot_roc_curve(
    y_val, lr_proba, "Logistic Regression", "Variant Pathogenicity",
    FIGURES_DIR / "02_lr_variants_roc_curve.png"
)

# Feature importance
lr_top_features = plot_feature_importance(
    lr_model, X_train_bal.columns, "Logistic Regression", "Variant Pathogenicity",
    FIGURES_DIR / "03_lr_variants_feature_importance.png"
)

print(" Visualizations saved")
print("\nTop 5 Features:")
for feat, imp in lr_top_features[-5:][::-1]:
    print(f"  {feat}: {imp:.4f}")

---
## 5. Task 1: Variant Pathogenicity - Decision Tree

In [ ]:
print("\nModel 2: Decision Tree")
print("-" * 60)

# Train on balanced data
dt_model = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=100,
    min_samples_leaf=50,
    random_state=RANDOM_STATE
)

print("Training...")
dt_model.fit(X_train_bal, y_train_bal)
print("Training complete")

# Evaluate
dt_results, dt_pred, dt_proba = evaluate_model(
    dt_model, X_train_bal, y_train_bal, X_val, y_val,
    "Decision Tree", "Variant Pathogenicity"
)

# Save model
dt_model_file = MODEL_DIR / "baseline_dt_variants.pkl"
with open(dt_model_file, 'wb') as f:
    pickle.dump(dt_model, f)
print(f"\n Model saved: {dt_model_file.name}")

In [ ]:
# Visualizations
print("\nGenerating visualizations...")

# Confusion matrix
plot_confusion_matrix(
    y_val, dt_pred, "Decision Tree", "Variant Pathogenicity",
    FIGURES_DIR / "04_dt_variants_confusion_matrix.png"
)

# ROC curve
plot_roc_curve(
    y_val, dt_proba, "Decision Tree", "Variant Pathogenicity",
    FIGURES_DIR / "05_dt_variants_roc_curve.png"
)

# Feature importance
dt_top_features = plot_feature_importance(
    dt_model, X_train_bal.columns, "Decision Tree", "Variant Pathogenicity",
    FIGURES_DIR / "06_dt_variants_feature_importance.png"
)

print(" Visualizations saved")
print("\nTop 5 Features:")
for feat, imp in dt_top_features[-5:][::-1]:
    print(f"  {feat}: {imp:.4f}")

---
## 6. Task 2: SV Risk - Logistic Regression

In [ ]:
print("\n" + "="*80)
print("TASK 2: STRUCTURAL VARIANT RISK PREDICTION")
print("="*80)
print("\nModel 1: Logistic Regression")
print("-" * 60)

# Train (no SMOTE needed - already balanced)
lr_sv_model = LogisticRegression(
    penalty='l2',
    C=1.0,
    max_iter=1000,
    random_state=RANDOM_STATE
)

print("Training...")
lr_sv_model.fit(X_sv_train, y_sv_train)
print(" Training complete")

# Evaluate
lr_sv_results, lr_sv_pred, lr_sv_proba = evaluate_model(
    lr_sv_model, X_sv_train, y_sv_train, X_sv_val, y_sv_val,
    "Logistic Regression", "SV Risk"
)

# Save model
lr_sv_file = MODEL_DIR / "baseline_lr_sv.pkl"
with open(lr_sv_file, 'wb') as f:
    pickle.dump(lr_sv_model, f)
print(f"\n Model saved: {lr_sv_file.name}")

In [ ]:
# Visualizations
print("\nGenerating visualizations...")

plot_confusion_matrix(
    y_sv_val, lr_sv_pred, "Logistic Regression", "SV Risk",
    FIGURES_DIR / "07_lr_sv_confusion_matrix.png"
)

plot_roc_curve(
    y_sv_val, lr_sv_proba, "Logistic Regression", "SV Risk",
    FIGURES_DIR / "08_lr_sv_roc_curve.png"
)

lr_sv_top_features = plot_feature_importance(
    lr_sv_model, X_sv_train.columns, "Logistic Regression", "SV Risk",
    FIGURES_DIR / "09_lr_sv_feature_importance.png", top_n=13
)

print(" Visualizations saved")

---
## 7. Task 2: SV Risk - Decision Tree

In [ ]:
print("\nModel 2: Decision Tree")
print("-" * 60)

dt_sv_model = DecisionTreeClassifier(
    max_depth=8,
    min_samples_split=50,
    min_samples_leaf=25,
    random_state=RANDOM_STATE
)

print("Training...")
dt_sv_model.fit(X_sv_train, y_sv_train)
print(" Training complete")

# Evaluate
dt_sv_results, dt_sv_pred, dt_sv_proba = evaluate_model(
    dt_sv_model, X_sv_train, y_sv_train, X_sv_val, y_sv_val,
    "Decision Tree", "SV Risk"
)

# Save model
dt_sv_file = MODEL_DIR / "baseline_dt_sv.pkl"
with open(dt_sv_file, 'wb') as f:
    pickle.dump(dt_sv_model, f)
print(f"\n Model saved: {dt_sv_file.name}")

In [ ]:
# Visualizations
print("\nGenerating visualizations...")

plot_confusion_matrix(
    y_sv_val, dt_sv_pred, "Decision Tree", "SV Risk",
    FIGURES_DIR / "10_dt_sv_confusion_matrix.png"
)

plot_roc_curve(
    y_sv_val, dt_sv_proba, "Decision Tree", "SV Risk",
    FIGURES_DIR / "11_dt_sv_roc_curve.png"
)

dt_sv_top_features = plot_feature_importance(
    dt_sv_model, X_sv_train.columns, "Decision Tree", "SV Risk",
    FIGURES_DIR / "12_dt_sv_feature_importance.png", top_n=13
)

print(" Visualizations saved")

---
## 8. Model Comparison

In [ ]:
# Create comparison table
comparison = pd.DataFrame([
    {
        'Task': 'Variant Pathogenicity',
        'Model': 'Logistic Regression',
        'F1': lr_results['validation']['f1'],
        'Precision': lr_results['validation']['precision'],
        'Recall': lr_results['validation']['recall'],
        'ROC-AUC': lr_results['validation']['roc_auc']
    },
    {
        'Task': 'Variant Pathogenicity',
        'Model': 'Decision Tree',
        'F1': dt_results['validation']['f1'],
        'Precision': dt_results['validation']['precision'],
        'Recall': dt_results['validation']['recall'],
        'ROC-AUC': dt_results['validation']['roc_auc']
    },
    {
        'Task': 'SV Risk',
        'Model': 'Logistic Regression',
        'F1': lr_sv_results['validation']['f1'],
        'Precision': lr_sv_results['validation']['precision'],
        'Recall': lr_sv_results['validation']['recall'],
        'ROC-AUC': lr_sv_results['validation']['roc_auc']
    },
    {
        'Task': 'SV Risk',
        'Model': 'Decision Tree',
        'F1': dt_sv_results['validation']['f1'],
        'Precision': dt_sv_results['validation']['precision'],
        'Recall': dt_sv_results['validation']['recall'],
        'ROC-AUC': dt_sv_results['validation']['roc_auc']
    }
])

print("\n" + "="*80)
print("BASELINE MODEL COMPARISON")
print("="*80)
print(comparison.to_string(index=False))

# Save comparison
comparison.to_csv(METRICS_DIR / "baseline_model_comparison.csv", index=False)
print(f"\n Comparison saved: baseline_model_comparison.csv")

---
## 9. Summary Report

In [ ]:
# Compile summary
summary = {
    'timestamp': datetime.now().isoformat(),
    'models_trained': 4,
    'variant_pathogenicity': {
        'logistic_regression': lr_results,
        'decision_tree': dt_results,
        'best_model': 'Logistic Regression' if lr_results['validation']['f1'] > dt_results['validation']['f1'] else 'Decision Tree',
        'best_f1': max(lr_results['validation']['f1'], dt_results['validation']['f1'])
    },
    'sv_risk': {
        'logistic_regression': lr_sv_results,
        'decision_tree': dt_sv_results,
        'best_model': 'Logistic Regression' if lr_sv_results['validation']['recall'] > dt_sv_results['validation']['recall'] else 'Decision Tree',
        'best_recall': max(lr_sv_results['validation']['recall'], dt_sv_results['validation']['recall'])
    }
}

# Save summary
summary_file = METRICS_DIR / "baseline_summary.json"
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"\nVariant Pathogenicity:")
print(f"  Best Model: {summary['variant_pathogenicity']['best_model']}")
print(f"  Best F1: {summary['variant_pathogenicity']['best_f1']:.4f}")
print(f"\nSV Risk:")
print(f"  Best Model: {summary['sv_risk']['best_model']}")
print(f"  Best Recall: {summary['sv_risk']['best_recall']:.4f}")

print(f"\n Summary saved: {summary_file.name}")

In [ ]:
# List all outputs
print("\n" + "="*80)
print("FILES CREATED")
print("="*80)

print(f"\nModels ({MODEL_DIR.relative_to(PROJECT_ROOT)}):")
for file in sorted(MODEL_DIR.glob("baseline_*.pkl")):
    print(f"  - {file.name}")

print(f"\nMetrics ({METRICS_DIR.relative_to(PROJECT_ROOT)}):")
for file in sorted(METRICS_DIR.glob("baseline_*")):
    print(f"  - {file.name}")

print(f"\nFigures ({FIGURES_DIR.relative_to(PROJECT_ROOT)}):")
for file in sorted(FIGURES_DIR.glob("*.png")):
    print(f"  - {file.name}")

print("\n" + "="*80)
print("PHASE 3B COMPLETE: BASELINE MODELS TRAINED")
print("="*80)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\nNext step: 03c_ensemble_models.ipynb")
print("  - Train Random Forest, XGBoost, LightGBM")
print("  - Hyperparameter tuning")
print("  - Beat baseline performance")
print("="*80)